In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.20, random_state=42, stratify=y)
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
score = accuracy_score(y_test, model.predict(X_test))
print(f"Single split accuracy: {score:.3f}")

Single split accuracy: 1.000


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"]).values
y = df["Pass"].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = [] 
for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    model = LogisticRegression(max_iter=500)
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_val, model.predict(X_val))
    fold_scores.append(acc)
    print(f" Fold {fold}: {acc:.3f}")
print(f"Mean: {np.mean(fold_scores):.3f} Std: {np.std(fold_scores):.3f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

 Fold 1: 1.000
 Fold 2: 1.000
 Fold 3: 1.000
 Fold 4: 1.000
 Fold 5: 1.000
Mean: 1.000 Std: 0.000


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"])
y = df["Pass"]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500))
])
scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
print(f"Accuracy per fold: {scores.round(3)}")
print(f"Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")

cv_results = cross_validate(
    pipe, X, y, cv=5,
    scoring=["accuracy", "f1"],
    return_train_score=True
)
for key, vals in cv_results.items():
    print(f"{key:30s}: {vals.round(3)}")
    
tree_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(max_depth=None))
])

cv_tree = cross_validate(tree_pipe, X, y, cv=5, scoring="accuracy", return_train_score=True)
print(f"Tree train: {cv_tree['train_score'].mean():.3f}")
print(f"Tree test : {cv_tree['test_score'].mean():.3f}")

Accuracy per fold: [1. 1. 1. 1. 1.]
Mean: 1.000 (+/- 0.000)
fit_time                      : [0.014 0.053 0.017 0.025 0.013]
score_time                    : [1.21  0.017 0.018 0.018 0.008]
test_accuracy                 : [1. 1. 1. 1. 1.]
train_accuracy                : [1. 1. 1. 1. 1.]
test_f1                       : [1. 1. 1. 1. 1.]
train_f1                      : [1. 1. 1. 1. 1.]
Tree train: 1.000
Tree test : 1.000


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.20, random_state=42, stratify=y)
 
pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", DecisionTreeClassifier(random_state=42))
])
param_grid = {
    "clf__max_depth": [2, 3, 4, 5, None],
    "clf__min_samples_split":[2, 5, 10],
    "clf__criterion": ["gini", "entropy"],
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1, 
    verbose=1,
    refit=True 
)
grid_search.fit(X_train, y_train)
print("Best params:", grid_search.best_params_)

print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(grid_search.cv_results_)
cols = ["param_clf__max_depth", "param_clf__min_samples_split", "mean_test_score", "std_test_score", "rank_test_score"]
print(results_df[cols].sort_values("rank_test_score").head(5))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: {'clf__criterion': 'gini', 'clf__max_depth': 2, 'clf__min_samples_split': 2}
Best CV accuracy: 1.000
Test accuracy: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00       132

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

  param_clf__max_depth  param_clf__min_samples_split  mean_test_score  \
0                    2                             2              1.0   
1                    2                             5              1.0   
2                    2                            10              1.0   
3                    3                             2              1.0   
4                    3                             5              1.0   

   std_test_score  rank_test_score  


In [5]:
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"])
y = df["Pass"]
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.20, random_state=42, stratify=y)

pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(random_state=42))
])

param_dist = {
    "clf__n_estimators": randint(50, 300),
    "clf__max_depth": [3, 4, 5, 6, None],
    "clf__max_features": uniform(0.3, 0.7), 
    "clf__min_samples_leaf": randint(1, 10),
}

rand_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=40, 
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        random_state=42,
        verbose=1,
        refit=True
)
rand_search.fit(X_train, y_train)
print("Best params:", rand_search.best_params_)
print(f"Best CV accuracy: {rand_search.best_score_:.3f}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params: {'clf__max_depth': 6, 'clf__max_features': np.float64(0.9655000144869412), 'clf__min_samples_leaf': 8, 'clf__n_estimators': 238}
Best CV accuracy: 1.000


In [8]:
print("MINI PROJECT \n")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("students_feature_engineered_v2.csv")
X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

base_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42))
])

base_score = cross_val_score(base_pipe, X_train, y_train, cv=5, scoring="accuracy")
print(f"Baseline CV: {base_score.mean():.3f} +/- {base_score.std():.3f}")

grid = GridSearchCV(base_pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1, refit=True)
param_grid = {
    "clf__max_depth": [2, 3, 4, 5],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}
grid = GridSearchCV(base_pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1, refit=True)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print(f"Tuned CV accuracy: {grid.best_score_:.3f}")

best = grid.best_estimator_
y_pred = best.predict(X_test)
print(f"\nFinal Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fail","Pass"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results = X_test.copy()
results["y_true"] = y_test.values
results["y_pred"] = y_pred
results.to_csv("test_predictions.csv", index=False)
print("Saved: test_predictions.csv")



MINI PROJECT 

Train: 800 rows | Test: 200 rows
Baseline CV: 1.000 +/- 0.000
Best params: {'clf__max_depth': 2, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Tuned CV accuracy: 1.000

Final Test Accuracy: 1.000

Classification Report:
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00        68
        Pass       1.00      1.00      1.00       132

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Confusion Matrix:
[[ 68   0]
 [  0 132]]
Saved: test_predictions.csv
